# 02 · Why random K-fold lies on this data

Two mechanisms inflate a shuffled split: card identity leaks across folds, and the model never has to extrapolate forward in time. This notebook measures both with the actual training frame, then shows the fold layout the project uses instead.

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
from fraudlake.config import get_settings
S = get_settings()
A = S.artifacts_dir
df = pd.read_parquet(A / "mart" / "training.parquet")
df["txn_ts"] = pd.to_datetime(df["txn_ts"])
print(f"{len(df):,} labelled transactions, {df.shape[1]} columns, fraud rate {df.is_fraud.mean():.3%}")

## 1. Card overlap between train and validation

Under a shuffled split almost every validation card has already been seen in training. Under the time split, a large share of validation cards are new, which is what production looks like.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from fraudlake.modeling.splits import time_holdout_split, time_series_folds
df = df.sort_values(["transaction_dt", "transaction_id"]).reset_index(drop=True)
y = df["is_fraud"].to_numpy()
rows = []
for i, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(df, y)):
    seen = df.iloc[va]["card_uid"].isin(set(df.iloc[tr]["card_uid"])).mean()
    rows.append(("shuffled K-fold", i, seen))
train_pos, hold_pos, cut = time_holdout_split(df["transaction_dt"], S.holdout_fraction)
train = df.iloc[train_pos].reset_index(drop=True)
for f in time_series_folds(train["transaction_dt"], S.n_folds, S.fold_gap_seconds):
    seen = train.iloc[f.val_idx]["card_uid"].isin(set(train.iloc[f.train_idx]["card_uid"])).mean()
    rows.append(("time folds", f.index, seen))
seen_hold = df.iloc[hold_pos]["card_uid"].isin(set(train["card_uid"])).mean()
rows.append(("time holdout", 0, seen_hold))
ov = pd.DataFrame(rows, columns=["scheme", "fold", "share of validation cards seen in training"])
ov.groupby("scheme")["share of validation cards seen in training"].mean().round(3)

## 2. The fold layout actually used

Expanding-window folds inside the first 80% of time, each with a one-day gap; the last 20% is never touched until `fraudlake evaluate`.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.8))
t0, t1 = df["txn_ts"].min(), df["txn_ts"].max()
for f in time_series_folds(train["transaction_dt"], S.n_folds, S.fold_gap_seconds):
    a, b = train.iloc[f.train_idx]["txn_ts"].agg(["min", "max"]); c, d = train.iloc[f.val_idx]["txn_ts"].agg(["min", "max"])
    ax.barh(f.index, (b - a).days, left=a, color="steelblue"); ax.barh(f.index, (d - c).days, left=c, color="orange")
ax.barh(S.n_folds, (t1 - df.iloc[hold_pos]["txn_ts"].min()).days, left=df.iloc[hold_pos]["txn_ts"].min(), color="crimson")
ax.set_yticks(range(S.n_folds + 1)); ax.set_yticklabels([f"fold {i}" for i in range(S.n_folds)] + ["holdout"])
ax.set(title="blue = fit, orange = validate (1-day gap before it), red = untouched holdout"); ax.invert_yaxis(); plt.tight_layout()

## 3. The measured optimism gap

`fraudlake evaluate` retrains the selected configuration with shuffled stratified K-fold on the same rows and features. The difference is what a naive pipeline would have reported on top of the truth.

In [ ]:
ev = json.loads((A / "evaluation" / "metrics.json").read_text())
vc = ev["validation_comparison"]
pd.DataFrame({"PR-AUC": [vc["random_kfold_pr_auc"], vc["time_cv_pr_auc"], vc["holdout_pr_auc"]],
              "±": [vc["random_kfold_pr_auc_std"], vc["time_cv_pr_auc_std"], np.nan]},
             index=["shuffled K-fold", "time folds (CV)", "time holdout"]).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
vals = [vc["random_kfold_pr_auc"], vc["time_cv_pr_auc"], vc["holdout_pr_auc"]]
ax.bar(["shuffled\nK-fold", "time folds", "time holdout"], vals, color=["gray", "steelblue", "crimson"])
for i, v in enumerate(vals): ax.text(i, v + 0.005, f"{v:.3f}", ha="center")
ax.set(ylabel="PR-AUC", title=f"optimism gap of a random split: {vc['optimism_gap']:+.3f}"); ax.set_ylim(0, max(vals) * 1.15); plt.tight_layout()

## 4. Adversarial validation: which features are clocks?

A classifier trained to tell training-window rows from holdout-window rows. High AUC means the feature distribution shifts with time; the top features by gain are the ones the model would use to learn *when* rather than *whether*.

In [ ]:
sel = json.loads((A / "features" / "selected.json").read_text())
print(f"AUC before dropping drift features: {sel['adversarial_auc_before']:.3f}   after: {sel['adversarial_auc_after']:.3f}")
top = pd.DataFrame(sel["adversarial_top"][:15], columns=["feature", "gain share"]).set_index("feature")
top.plot.barh(figsize=(7, 4.5), color="darkorange", legend=False); plt.gca().invert_yaxis(); plt.title("train-vs-holdout classifier: top features by gain"); plt.tight_layout()
drops = pd.Series(sel["dropped"]).str.split("(").str[0].str.split("=").str[0].value_counts()
drops.rename("features dropped, by filter")